<a href="https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [39]:
# SETUP and SECHEMA CHECK

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzad-jatoi/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd, json, os

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{WAREHOUSE}/dim_content.parquet') LIMIT 1").df().to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule (plain words):** flag a page as a refresh candidate if it's stale (no update in a while) AND getting real search visibility (impressions above a floor) but converting poorly (low CTR relative to its position tier). Two signals back this: staleness (behind FlyRank's real refresh flags) and CTR-vs-position (behind the CTR-fix logic).

**Reason code:** `STALE_LOW_CTR`
**Action label:** `refresh_priority`

In [40]:
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
DIM_CLIENTS = f"{WAREHOUSE}/dim_clients.parquet"
FACT_MONTH = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/data_0.parquet"
print("Paths set.")

Paths set.


In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: staleness — bucket and verdict
sig1 = con.sql(f"""
    SELECT
        CASE
            WHEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') < 90 THEN '0-90d'
            WHEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') < 180 THEN '90-180d'
            WHEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') < 365 THEN '180-365d'
            ELSE '365d+'
        END as staleness_bucket,
        COUNT(*) as n,
        AVG(f.gsc_avg_position) as avg_position
    FROM read_parquet('{DIM_CONTENT}') c
    JOIN read_parquet('{FACT_MONTH}') f
      ON c.content_hash_id = f.content_hash_id
    WHERE f.gsc_data_available IS TRUE AND c.is_deleted IS NOT TRUE
    GROUP BY 1
    ORDER BY 1
""").df()
print(sig1)
print("Verdict: CONFIRMED if staler buckets show worse (higher) avg_position")

# Signal 2: CTR vs position tier
sig2 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END as position_tier,
        COUNT(*) as n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM read_parquet('{FACT_MONTH}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY 1
""").df()
print(sig2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket        n  avg_position
0            0-90d  3598206     15.822003
1         180-365d     1416     18.588990
2          90-180d     9622     18.861998
Verdict: CONFIRMED if staler buckets show worse (higher) avg_position


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier        n   avg_ctr
0           1-3   727362  0.004756
1         11-20   519223  0.002770
2           20+   908354  0.001289
3          4-10  1456122  0.003473


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs("work/outputs", exist_ok=True)

scored = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id,
               AVG(gsc_impressions) as avg_impressions,
               AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) as avg_ctr,
               AVG(gsc_avg_position) as avg_position
        FROM read_parquet('{FACT_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT a.content_hash_id,
           DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') as days_since_update,
           a.avg_impressions, a.avg_ctr, a.avg_position,
           (a.avg_impressions *
            (DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')/365.0) /
            (a.avg_ctr + 0.005)) as score,
           'STALE_LOW_CTR' as reason_code,
           'refresh_priority' as action
    FROM agg a
    JOIN read_parquet('{DIM_CONTENT}') c ON a.content_hash_id = c.content_hash_id
    WHERE DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') >= 90
      AND a.avg_impressions >= 20
      AND c.is_deleted IS NOT TRUE
    ORDER BY score DESC
""").df()

scored.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(scored)} ranked rows.")
scored.head(10)

Wrote 59 ranked rows.


,content_hash_id,days_since_update,avg_impressions,avg_ctr,avg_position,score,reason_code,action
0,content_097459d155cccb26,124,1223.548387,0.001265,17.550666,66348.518893,STALE_LOW_CTR,refresh_priority
1,content_ac4e2d9d3bbb06de,124,1075.741935,0.000819,23.327797,62800.911611,STALE_LOW_CTR,refresh_priority
2,content_66d1fffc91f4f029,124,1001.225806,0.001689,15.216861,50849.440050,STALE_LOW_CTR,refresh_priority
3,content_f2df5a8a9057783e,124,1160.645161,0.003050,15.542928,48979.804154,STALE_LOW_CTR,refresh_priority
4,content_b956947c822af734,124,725.870968,0.000295,36.484478,46571.078673,STALE_LOW_CTR,refresh_priority
5,content_9598a57544925111,123,776.322581,0.001221,12.873404,42050.053293,STALE_LOW_CTR,refresh_priority
6,content_47da45b084a73115,124,476.580645,0.000179,50.756106,31260.907238,STALE_LOW_CTR,refresh_priority
7,content_b361694d518f80e2,124,583.967742,0.002472,16.775321,26551.771150,STALE_LOW_CTR,refresh_priority
8,content_0d2aaf57d7146812,123,679.354839,0.003747,9.765313,26174.074706,STALE_LOW_CTR,refresh_priority
9,content_dfbc1b6a0f68e28c,124,512.225806,0.002491,23.107159,23228.956371,STALE_LOW_CTR,refresh_priority


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = scored.head(20).copy()
top20["what_would_make_it_wrong"] = "if the page's impressions are seasonal/one-off rather than a sustained visibility gap, refreshing it wastes editorial effort"

for i, row in top20.iterrows():
    print(f"#{i+1} | content: {row['content_hash_id'][:12]}... | action: {row['action']} | "
          f"reason: {row['reason_code']} | score: {row['score']:.1f} | "
          f"avg_position: {row['avg_position']:.1f} | avg_ctr: {row['avg_ctr']:.3f} | "
          f"stale: {row['days_since_update']}d")
    print(f"    -> what would make it wrong: {row['what_would_make_it_wrong']}")

top20[["content_hash_id","days_since_update","avg_impressions","avg_ctr","avg_position","score","reason_code","action"]]


#1 | content: content_0974... | action: refresh_priority | reason: STALE_LOW_CTR | score: 66348.5 | avg_position: 17.6 | avg_ctr: 0.001 | stale: 124d
    -> what would make it wrong: if the page's impressions are seasonal/one-off rather than a sustained visibility gap, refreshing it wastes editorial effort
#2 | content: content_ac4e... | action: refresh_priority | reason: STALE_LOW_CTR | score: 62800.9 | avg_position: 23.3 | avg_ctr: 0.001 | stale: 124d
    -> what would make it wrong: if the page's impressions are seasonal/one-off rather than a sustained visibility gap, refreshing it wastes editorial effort
#3 | content: content_66d1... | action: refresh_priority | reason: STALE_LOW_CTR | score: 50849.4 | avg_position: 15.2 | avg_ctr: 0.002 | stale: 124d
    -> what would make it wrong: if the page's impressions are seasonal/one-off rather than a sustained visibility gap, refreshing it wastes editorial effort
#4 | content: content_f2df... | action: refresh_priority | reason: STALE_LOW

,content_hash_id,days_since_update,avg_impressions,avg_ctr,avg_position,score,reason_code,action
0,content_097459d155cccb26,124,1223.548387,0.001265,17.550666,66348.518893,STALE_LOW_CTR,refresh_priority
1,content_ac4e2d9d3bbb06de,124,1075.741935,0.000819,23.327797,62800.911611,STALE_LOW_CTR,refresh_priority
2,content_66d1fffc91f4f029,124,1001.225806,0.001689,15.216861,50849.440050,STALE_LOW_CTR,refresh_priority
3,content_f2df5a8a9057783e,124,1160.645161,0.003050,15.542928,48979.804154,STALE_LOW_CTR,refresh_priority
4,content_b956947c822af734,124,725.870968,0.000295,36.484478,46571.078673,STALE_LOW_CTR,refresh_priority
5,content_9598a57544925111,123,776.322581,0.001221,12.873404,42050.053293,STALE_LOW_CTR,refresh_priority
6,content_47da45b084a73115,124,476.580645,0.000179,50.756106,31260.907238,STALE_LOW_CTR,refresh_priority
7,content_b361694d518f80e2,124,583.967742,0.002472,16.775321,26551.771150,STALE_LOW_CTR,refresh_priority
8,content_0d2aaf57d7146812,123,679.354839,0.003747,9.765313,26174.074706,STALE_LOW_CTR,refresh_priority
9,content_dfbc1b6a0f68e28c,124,512.225806,0.002491,23.107159,23228.956371,STALE_LOW_CTR,refresh_priority


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Weak pick check: any top-10 picks resting on a thin impressions sample?
weak = scored.head(10)[scored.head(10)["avg_impressions"] < 100]
print(f"Weak picks in top 10 (avg_impressions < 100, less reliable): {len(weak)}")
if len(weak) > 0:
    print(weak[["content_hash_id", "avg_impressions", "avg_ctr", "score"]])

# Leakage check: confirm every input is pre-decision and within the March window only
used_cols = {"content_hash_id", "days_since_update", "avg_impressions", "avg_ctr", "avg_position"}
print(f"\nColumns used in scoring: {used_cols}")
print("All derived from month=2026-03 (fact table) and content_updated_date (dim_content) only.")
print("No columns from a later month, no trend/outcome labels, no product/CRM flags were used.")

# Save metrics receipt — this JSON IS committed (unlike the CSV)
metrics = {
    "n_ranked": len(scored),
    "n_weak_top10": len(weak),
    "signal1_staleness_verdict": "MIXED",
    "signal2_ctr_position_verdict": "CONFIRMED",
    "month_used": "2026-03",
    "min_impressions_threshold": 20,   # <- update this to match your new loosened threshold
    "min_staleness_days": 90           # <- update this to match your new loosened threshold
}
with open("work/outputs/w04_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nSaved metrics:", metrics)


Weak picks in top 10 (avg_impressions < 100, less reliable): 0

Columns used in scoring: {'avg_position', 'avg_ctr', 'days_since_update', 'avg_impressions', 'content_hash_id'}
All derived from month=2026-03 (fact table) and content_updated_date (dim_content) only.
No columns from a later month, no trend/outcome labels, no product/CRM flags were used.

Saved metrics: {'n_ranked': 59, 'n_weak_top10': 0, 'signal1_staleness_verdict': 'MIXED', 'signal2_ctr_position_verdict': 'CONFIRMED', 'month_used': '2026-03', 'min_impressions_threshold': 20, 'min_staleness_days': 90}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.